Send Forecast Email
Reads the latest generated forecast and emails a readable summary to the ERP team for validation.

**Input**: gold/live/forecasts/overall_forecast_latest.json
**Output**: email sent

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service
import json
import datetime

blob_service = get_blob_service(storage_account_name, storage_account_key)

to_emails = ["erp-person@brownsgroup.com"]  # add more addresses to this list if needed

Load active overall forecasts

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/battery/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/battery/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly = json.loads(stream)

print(active_weekly)
print(active_monthly)

Load active brand forecasts

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/battery/active/brand_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_brand_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/battery/active/brand_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_brand_monthly = json.loads(stream)

print(active_brand_weekly)
print(active_brand_monthly)

Build the email body

In [0]:
body = """Hello,

Here is the latest battery sales forecast.

WEEKLY FORECAST
"""
for w in active_weekly:
    body += f"Week starting {w['week_start']}: {w['predicted_units']:,} units (generated {w['generated_date']})\n"

body += "\nMONTHLY FORECAST\n"
for m in active_monthly:
    body += f"{m['month_start']}: {m['predicted_units']:,} units (range: {m['lower_bound']:,} - {m['upper_bound']:,}, generated {m['generated_date']})\n"

body += "\nBRAND BREAKDOWN (Weekly)\n"
for w in active_brand_weekly:
    body += f"{w['brand_code']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nBRAND BREAKDOWN (Monthly)\n"
for m in active_brand_monthly:
    body += f"{m['brand_code']} — {m['month_start']}: {m['predicted_units']:,} units\n"

body += """
Please review and let us know if these numbers look reasonable based on your knowledge of current orders/promotions.

This is an automated message from the Exide Sales Forecasting pipeline.
"""

print(body)

Excel attachments

In [0]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import smtplib

today_str = datetime.date.today().isoformat()

msg = MIMEMultipart()
msg["From"] = smtp_username
msg["To"] = ", ".join(to_emails)
msg["Subject"] = f"Battery Sales Forecast — {today_str}"
msg.attach(MIMEText(body, "plain"))

def attach_excel_from_blob(msg, blob_service, blob_path, filename):
    blob_client = blob_service.get_blob_client(container="gold", blob=blob_path)
    excel_bytes = blob_client.download_blob().readall()
    attachment = MIMEApplication(excel_bytes, _subtype="xlsx")
    attachment.add_header("Content-Disposition", "attachment", filename=filename)
    msg.attach(attachment)

attach_excel_from_blob(msg, blob_service, "live/forecasts/battery/history/weekly_forecast_history.xlsx", "weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, "live/forecasts/battery/history/monthly_forecast_history.xlsx", "monthly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, "live/forecasts/battery/history/brand_weekly_forecast_history.xlsx", "brand_weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, "live/forecasts/battery/history/brand_monthly_forecast_history.xlsx", "brand_monthly_forecast_history.xlsx")

try:
    server = smtplib.SMTP(smtp_server, smtp_port)
    server.starttls()
    server.login(smtp_username, smtp_password)
    server.sendmail(smtp_username, to_emails, msg.as_string())
    print(f"Email sent to {', '.join(to_emails)}")
except Exception as e:
    print(f"Failed to send email: {e}")
finally:
    server.quit()